In [12]:
# Cell 1: Import Libraries
import pandas as pd
import json

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [13]:
# Cell 2: Load Files
ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

print("Ledger shape:", ledger.shape)
print("Gateway shape:", gateway.shape)
print("\nLedger columns:", ledger.columns.tolist())
print("\nGateway columns:", gateway.columns.tolist())

Ledger shape: (10, 6)
Gateway shape: (9, 6)

Ledger columns: ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']

Gateway columns: ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']


In [15]:
# Cell 3: Check Duplicates and Nulls
print("=== LEDGER ===")
print("Duplicates:", ledger.duplicated().sum())
print("Nulls:\n", ledger.isnull().sum())

print("\n=== GATEWAY ===")
print("Duplicates:", gateway.duplicated().sum())
print("Nulls:\n", gateway.isnull().sum())

=== LEDGER ===
Duplicates: 0
Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

=== GATEWAY ===
Duplicates: 0
Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [14]:
# Cell 4: Records missing in gateway
missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])]
print("Missing in gateway:", len(missing_in_gateway))
print(missing_in_gateway)
missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)

Missing in gateway: 2
  transaction_id transaction_date merchant_id  amount_usd   status  \
3           R004       2026-03-02        M003      2100.0  success   
9           R010       2026-03-05        M004      2500.0  success   

  payment_method  
3           Card  
9         Wallet  


In [16]:
# Cell 5: Records missing in ledger
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger['transaction_id'])]
print("Missing in ledger:", len(missing_in_ledger))
print(missing_in_ledger)
missing_in_ledger.to_csv('missing_in_ledger.csv', index=False)

Missing in ledger: 1
  transaction_id transaction_date merchant_id  amount_usd   status  \
8           R011       2026-03-05        M003      1800.0  success   

  payment_method  
8           Card  


In [17]:
# Cell 6: Amount Mismatches
merged = ledger.merge(gateway, on='transaction_id', suffixes=('_ledger', '_gateway'))
amount_mismatches = merged[merged['amount_usd_ledger'] != merged['amount_usd_gateway']]
print("Amount mismatches:", len(amount_mismatches))
print(amount_mismatches[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway']])
amount_mismatches.to_csv('amount_mismatches.csv', index=False)

Amount mismatches: 2
  transaction_id  amount_usd_ledger  amount_usd_gateway
1           R002              850.0               900.0
6           R008              640.0               600.0


In [18]:
# Cell 7: Status Mismatches
status_mismatches = merged[merged['status_ledger'] != merged['status_gateway']]
print("Status mismatches:", len(status_mismatches))
print(status_mismatches[['transaction_id', 'status_ledger', 'status_gateway']])
status_mismatches.to_csv('status_mismatches.csv', index=False)

Status mismatches: 1
  transaction_id status_ledger status_gateway
3           R005       success         failed


In [19]:
# Cell 8: Final Reconciliation Report
merged['reconciliation_status'] = 'matched'
merged.loc[merged['amount_usd_ledger'] != merged['amount_usd_gateway'], 'reconciliation_status'] = 'amount_mismatch'
merged.loc[merged['status_ledger'] != merged['status_gateway'], 'reconciliation_status'] = 'status_mismatch'

reconciliation_report = merged[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'status_ledger', 'status_gateway', 'reconciliation_status']]
print(reconciliation_report)
reconciliation_report.to_csv('reconciliation_report.csv', index=False)
print("\nReconciliation report saved!")

  transaction_id  amount_usd_ledger  amount_usd_gateway status_ledger  \
0           R001             1200.0              1200.0       success   
1           R002              850.0               900.0       success   
2           R003              500.0               500.0       success   
3           R005             7200.0              7200.0       success   
4           R006              950.0               950.0       success   
5           R007             3300.0              3300.0        failed   
6           R008              640.0               600.0       success   
7           R009             4100.0              4100.0       success   

  status_gateway reconciliation_status  
0        success               matched  
1        success       amount_mismatch  
2        success               matched  
3         failed       status_mismatch  
4        success               matched  
5         failed               matched  
6        success       amount_mismatch  
7        succe

In [20]:
# Cell 9: Summary Metrics
import json

summary = {
    "total_ledger_rows": len(ledger),
    "total_gateway_rows": len(gateway),
    "missing_in_gateway_count": len(missing_in_gateway),
    "missing_in_ledger_count": len(missing_in_ledger),
    "amount_mismatch_count": len(amount_mismatches),
    "status_mismatch_count": len(status_mismatches),
    "reconciliation_issue_count": len(amount_mismatches) + len(status_mismatches),
    "ledger_total_amount": round(ledger['amount_usd'].sum(), 2),
    "gateway_total_amount": round(gateway['amount_usd'].sum(), 2),
    "amount_at_risk": round(amount_mismatches['amount_usd_ledger'].sum(), 2)
}

with open('summary_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSummary metrics saved!")

{
  "total_ledger_rows": 10,
  "total_gateway_rows": 9,
  "missing_in_gateway_count": 2,
  "missing_in_ledger_count": 1,
  "amount_mismatch_count": 2,
  "status_mismatch_count": 1,
  "reconciliation_issue_count": 3,
  "ledger_total_amount": 23340.0,
  "gateway_total_amount": 20550.0,
  "amount_at_risk": 1490.0
}

Summary metrics saved!


In [23]:
# Cell 10: JSON Normalization
import json
import pandas as pd

with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)

df_api = pd.json_normalize(
    api_data['batches'],
    sep='_'
)

df_api.columns = df_api.columns.str.lower().str.replace('.', '_', regex=False)
print("API normalized shape:", df_api.shape)
print(df_api.head())
df_api.to_csv('api_normalized.csv', index=False)
print("\nAPI normalized saved!")

API normalized shape: (2, 5)
  batch_id                                        settlements  \
0     B001  [{'settlement_id': 'S001', 'amount_usd': 1520....   
1     B002  [{'settlement_id': 'S004', 'amount_usd': 2100....   

  merchant_merchant_id merchant_merchant_name merchant_region  
0                 M001             Alpha Mart            APAC  
1                 M004          Delta Travels              US  

API normalized saved!


In [24]:
# Cell 11: Dashboard Output CSVs
import pandas as pd

cleaned = pd.read_csv('cleaned_transactions.csv') \
    if 'cleaned_transactions.csv' in __import__('os').listdir() \
    else pd.DataFrame()

# Daily Summary
daily_summary = ledger.groupby('transaction_date').agg(
    total_gmv=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count')
).reset_index()
daily_summary.to_csv('daily_summary.csv', index=False)
print("Daily summary saved!")
print(daily_summary)

# Payment Method Breakdown
payment_breakdown = ledger.groupby('payment_method').agg(
    total_amount=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count')
).reset_index()
payment_breakdown.to_csv('payment_method_breakdown.csv', index=False)
print("\nPayment method breakdown saved!")
print(payment_breakdown)

# Region Breakdown
region_breakdown = ledger.groupby('merchant_id').agg(
    total_amount=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count')
).reset_index()
region_breakdown.to_csv('region_breakdown.csv', index=False)
print("\nRegion breakdown saved!")
print(region_breakdown)

# Merchant Performance Summary
merchant_summary = ledger.groupby('merchant_id').agg(
    total_gmv=('amount_usd', 'sum'),
    total_transactions=('transaction_id', 'count'),
    avg_amount=('amount_usd', 'mean')
).reset_index()
merchant_summary.to_csv('merchant_performance_summary.csv', index=False)
print("\nMerchant performance summary saved!")
print(merchant_summary)

Daily summary saved!
  transaction_date  total_gmv  transaction_count
0       2026-03-01     2050.0                  2
1       2026-03-02     2600.0                  2
2       2026-03-03     8150.0                  2
3       2026-03-04     3940.0                  2
4       2026-03-05     6600.0                  2

Payment method breakdown saved!
  payment_method  total_amount  transaction_count
0           Card       14890.0                  5
1     NetBanking        3300.0                  1
2            UPI        2150.0                  2
3         Wallet        3000.0                  2

Region breakdown saved!
  merchant_id  total_amount  transaction_count
0        M001        2340.0                  3
1        M002        5900.0                  3
2        M003        2100.0                  1
3        M004        9700.0                  2
4        M005        3300.0                  1

Merchant performance summary saved!
  merchant_id  total_gmv  total_transactions   avg_amount


In [25]:
# Cell 12: Verify all output files
import os

files_needed = [
    'missing_in_gateway.csv',
    'missing_in_ledger.csv',
    'amount_mismatches.csv',
    'status_mismatches.csv',
    'reconciliation_report.csv',
    'api_normalized.csv',
    'daily_summary.csv',
    'payment_method_breakdown.csv',
    'region_breakdown.csv',
    'merchant_performance_summary.csv',
    'summary_metrics.json'
]

print("=== FILE CHECK ===")
for f in files_needed:
    if f in os.listdir():
        print(f"✅ {f}")
    else:
        print(f"❌ MISSING: {f}")

=== FILE CHECK ===
✅ missing_in_gateway.csv
✅ missing_in_ledger.csv
✅ amount_mismatches.csv
✅ status_mismatches.csv
✅ reconciliation_report.csv
✅ api_normalized.csv
✅ daily_summary.csv
✅ payment_method_breakdown.csv
✅ region_breakdown.csv
✅ merchant_performance_summary.csv
✅ summary_metrics.json
